# Initialize the Master Data
Pull optimized ingestion data into the current notebook session.

In [0]:
base_df = spark.table("backblaze_master_delta")

# Engineer 7-Day Lag Features
We want to establish what the drive's health profile looked like exactly one week ago. We focus on SMART 5 (Reallocated Sectors) and SMART 187 (Uncorrectable Errors) as they are highly volatile pre-failure indicators.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

w = Window.partitionBy("serial_number").orderBy("date")

lag_df = base_df.withColumn(
    "smart_5_lag_7", F.coalesce(F.lag("smart_5_raw", 7).over(w), F.lit(0))
).withColumn(
    "smart_187_lag_7", F.coalesce(F.lag("smart_187_raw", 7).over(w), F.lit(0))
)

display(lag_df)

date,serial_number,model,capacity_bytes,failure,smart_5_raw,smart_9_raw,smart_187_raw,smart_188_raw,smart_197_raw,smart_198_raw,label,smart_5_lag_7,smart_187_lag_7
2025-10-01,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0
2025-10-02,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0
2025-10-03,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0
2025-10-04,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0
2025-10-05,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0
2025-10-06,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0
2025-10-07,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0
2025-10-08,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0
2025-10-09,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0
2025-10-10,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0


# Engineer Velocity (Delta) Features
Now, calculate the rate of change over that 7-day window. If this delta spikes sharply, it provides a massive signal for your Gradient Boosted Trees model to exploit.$$\text{Delta}_7 = \text{Current Raw Value} - \text{Value}_7\text{ Days Ago}$$

In [0]:
engineered_df = lag_df.withColumn(
    "smart_5_delta_7", F.col("smart_5_raw") - F.col("smart_5_lag_7")
).withColumn(
    "smart_187_delta_7", F.col("smart_187_raw") - F.col("smart_187_lag_7")
)

display(engineered_df)

date,serial_number,model,capacity_bytes,failure,smart_5_raw,smart_9_raw,smart_187_raw,smart_188_raw,smart_197_raw,smart_198_raw,label,smart_5_lag_7,smart_187_lag_7,smart_5_delta_7,smart_187_delta_7
2025-10-01,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0,0,0
2025-10-02,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0,0,0
2025-10-03,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0,0,0
2025-10-04,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0,0,0
2025-10-05,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0,0,0
2025-10-06,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0,0,0
2025-10-07,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0,0,0
2025-10-08,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0,0,0
2025-10-09,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0,0,0
2025-10-10,063ea317eea30010,DELLBOSS VD,480036847616,0,0,0,0,0,0,0,0,0,0,0,0


# Write to a Serverless Feature Table
Finally, persist these newly engineered columns into a standalone managed table so your modeling notebook can access them instantly without re-running the heavy time-series window functions.

In [0]:
engineered_df.write.format("delta").mode("overwrite").saveAsTable("backblaze_features_delta")